# Checkpointing Optimization with CheckpointObserver


The `CheckpointObserver` saves pickled `OptimizationState` snapshots to disk at configurable intervals. Useful for long optimization runs where you want to guard against interruption.


In [ ]:
import numpy as np
import tempfile, os
from optiland import optic
from optiland.optimization import minimize, OptimizationProblem, CheckpointObserver

lens = optic.Optic()
lens.surfaces.add(index=0, thickness=np.inf)
lens.surfaces.add(index=1, thickness=6, radius=25, material="N-BK7", is_stop=True)
lens.surfaces.add(index=2, thickness=3, radius=-25, material="N-SF11")
lens.surfaces.add(index=3, thickness=45, radius=-100)
lens.surfaces.add(index=4)
lens.set_aperture(aperture_type="EPD", value=10)
lens.fields.set_type("angle")
lens.fields.add(y=0.0)
lens.fields.add(y=0.7)
lens.wavelengths.add(value=0.5876, is_primary=True)
lens.update_paraxial()


In [ ]:
problem = OptimizationProblem()
for field in lens.fields.get_field_coords():
    input_data = {"optic": lens, "surface_number": -1, "Hx": field[0], "Hy": field[1],
                  "num_rays": 5, "wavelength": 0.5876, "distribution": "hexapolar"}
    problem.add_operand("rms_spot_size", target=0, weight=1, input_data=input_data)
problem.add_variable(lens, "radius", surface_number=1)
problem.add_variable(lens, "radius", surface_number=2)
problem.add_variable(lens, "radius", surface_number=3)
problem.add_variable(lens, "thickness", surface_number=3)


Pass a `CheckpointObserver` to save state snapshots every N accepted iterations. The constructor takes a `directory` path and an `every` interval (default 10).


In [ ]:
checkpoint_dir = os.path.join(tempfile.gettempdir(), "optiland_checkpoints")
ckpt = CheckpointObserver(directory=checkpoint_dir, every=5)
result = minimize(problem, "dls", observers=[ckpt])
print(result)
print(f"Checkpoint directory: {checkpoint_dir}")
print(f"Checkpoint files written: {len(ckpt._written)}")


Each checkpoint file is a pickled `OptimizationState` named `checkpoint_iter{N:06d}.pkl` (plus a `_final` snapshot written at the end of the run). Load a checkpoint with `pickle.load()` to inspect or resume from an intermediate state.
